In [ ]:
# 0. 패키지 버전 맞추기 — Colab에서 처음 한 번: 이 셀 실행 → 런타임 다시 시작 → Run All
import sys
if 'google.colab' in sys.modules:
    !pip install -q pandas==3.1.0 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 xgboost==3.4.1 lightgbm==4.7.0 pyarrow==26.0.0 PyYAML==6.0.3
    print('설치 끝: 처음이면 런타임 → 세션 다시 시작 후 Run All')
else:
    print('로컬: 설치 생략')

# pairwise 쌍 방식 확인 — 전체 쌍 vs 표본 8개 × 시드 5개 (Colab GPU)

- **왜**: 2026-11-16 비교(Fold 16, 입력 [Size, B/M, Mom, σ])에서 표본 8개가 전체 쌍과 0.0008 차이, 3.4배 빠름. 지금은 입력이 26개+RMW라 다시 확인한다.
- **조건**: 주 세트(기본+H1+H3+H2+H4, RMW 포함), XGB pairwise, 깊이 3, 학습률 0.01, L2 0, 16 Fold(192개월), 트리 해마다 선택과 500 고정(같은 학습). 결측 평균 채움 등 전처리는 basefeat 실행과 같다.
- **비교**: 전체 쌍(`topk`, 1000) 1번 vs 표본 8개(`mean`, 8) 시드 0~4.
- **판단**: 시드 간 흔들림과 전체 쌍과의 차이가 모두 0.002보다 충분히 작으면 튜닝·본 실행을 표본 8개(seed 0)로.
- **중간 저장**: 방식마다 `model_results/<AUTHOR>/pairwise_check_v1_parts/`에 저장, 끊기면 Run All로 이어감. 예상 약 1.5시간(전체 쌍 약 35분 + 표본 5번 약 50분).

In [ ]:
import shutil, subprocess
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
else:
    raise RuntimeError('GPU 런타임이 아니다(XGBoost는 cuda로 돌린다)')

## 1. 설정 (Drive 연결·코드 받기)

In [ ]:
import os, sys, subprocess
from datetime import datetime
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/JiunShin1214/finance_analytics.git'
GIT_REF = 'jiun'                # 브랜치 이름 — 그 브랜치의 GitHub 최신 코드
AUTHOR = 'jiun'                 # 결과 폴더 model_results/<AUTHOR>/ (영문·숫자·_·-만)
RUN_NAME = 'pairwise_check_v1'        # 같은 이름으로 다시 Run All 하면 Drive에 저장된 단위를 이어 씀
DRIVE_BASE = Path('/content/drive/MyDrive/금융데이터인텔리전스')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not GIT_REF:
        raise RuntimeError("GIT_REF에 브랜치 이름을 적는다.")
    ROOT = Path('/content/finance_analytics')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(ROOT)], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', '--prune'], check=True)
    _remote = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', '--verify', '--quiet', f'origin/{GIT_REF}'],
                             capture_output=True, text=True).stdout.strip()
    subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', _remote or GIT_REF], check=True)
    print('받은 코드:', GIT_REF, '@', subprocess.run(['git', '-C', str(ROOT), 'rev-parse', '--short', 'HEAD'],
                                               capture_output=True, text=True, check=True).stdout.strip())
    if not (ROOT / 'data').is_symlink():       # 저장소 data/를 Drive 공유 data로 연결
        (ROOT / 'data').rename(ROOT / 'data_repo_placeholder')
        (ROOT / 'data').symlink_to(DRIVE_BASE / 'data')
    OUT_ROOT = Path('/content/drive/MyDrive/model_results')   # 중간 저장(parts)과 결과 모두 내 드라이브 → 세션이 끊겨도 남음
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir() and (p / 'config' / 'experiment.yaml').is_file())
    OUT_ROOT = None

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import yaml
import pandas as pd
from src.evaluation import model_test as mt
from src.evaluation import base_model_runs as bmr
from src.features.h3 import compute_h3_sigma_grid

config = yaml.safe_load((ROOT / 'config' / 'experiment.yaml').read_text(encoding='utf-8'))
mt.check_no_test(config)
need = [config['PATHS'][k] for k in ('panel', 'eligibility', 'h1_components', 'base_features', 'h2', 'h4')]
missing = [n for n in need if not (ROOT / n).is_file()]
if missing:
    raise FileNotFoundError('Drive data/에 없음: ' + ', '.join(missing))
STARTED = datetime.now()
PARTS = bmr.parts_dir(ROOT, config, author=AUTHOR, run_name=RUN_NAME, out_root=OUT_ROOT)
print('중간 저장 폴더:', PARTS)
print('이미 저장된 단위:', len(list(PARTS.glob('*.pkl'))))

## 2. H3 격자·전처리

In [ ]:
panel = pd.read_parquet(ROOT / config['PATHS']['panel'], columns=['stock_id', 'month', 'return', 'eligible'])
eligibility = pd.read_parquet(ROOT / config['PATHS']['eligibility'])
H3_GRID = compute_h3_sigma_grid(panel, eligibility)
del panel, eligibility
PREP = bmr.prepare(ROOT, config, 'boosting', input_transform='rank', h3_grid=H3_GRID)

## 3. 실행 (6단위)

In [ ]:
SET = 'base+h1+h3+h2+h4'
COLS = bmr.input_sets('boosting')[SET]
assert 'rmw' in COLS, 'RMW가 없는 코드다(GIT_REF 확인)'
SETTING = [('xgb_pairwise', 'decile', 'pairwise', 3, 0.01)]   # 깊이 3, 학습률 0.01, L2 0
MEAN8 = {'lambdarank_pair_method': 'mean', 'lambdarank_num_pair_per_sample': 8}
VARIANTS = {'full': None, **{f'mean8_seed{s}': {**MEAN8, 'seed': s} for s in range(5)}}
fp = bmr.fingerprint(ROOT, config, 'boosting', 'rank')
units = [(f'pairwise_check__{name}', {'set': SET, 'setting': SETTING, 'overrides': ov, 'fixed_trees': [500]},
          (lambda ov=ov: mt.run_boosting(PREP, config, 'xgb_pairwise', phase='basefeat', settings=SETTING, sets={SET: COLS},
                                         overrides=ov, fixed_trees=[500])))
         for name, ov in VARIANTS.items()]
RESULTS = bmr.run_units(units, PARTS, fp)

## 4. 요약

In [ ]:
import numpy as np
rows, monthly = [], {}
for name, r in RESULTS.items():
    v = name.split('__')[1]
    vk = r['val_keys']
    for key, pred in r['predictions'].items():
        trees = '500 고정' if key.endswith('_t500') else '해마다'
        ic = mt.monthly_ic(vk.target_month.to_numpy(), pred, vk.y.to_numpy()).ic
        monthly[(v, trees)] = ic
        rows.append({'방식': v, '트리 수': trees, '평균 IC': ic.mean(), 'ICIR': ic.mean() / ic.std(ddof=1), '개월': int(ic.notna().sum()),
                     '학습 시간(분)': r['timings'].seconds.sum() / 60})
T = pd.DataFrame(rows).sort_values(['트리 수', '방식']).reset_index(drop=True)
print(T.round(4).to_string())
for trees in ('해마다', '500 고정'):
    seeds = T[(T['트리 수'] == trees) & T['방식'].str.startswith('mean8')]['평균 IC']
    full = T[(T['트리 수'] == trees) & (T['방식'] == 'full')]['평균 IC'].iloc[0]
    print(f"[{trees}] 시드 5개 평균 {seeds.mean():.4f}, 시드 간 표준편차 {seeds.std(ddof=1):.4f}, 범위 {seeds.min():.4f}~{seeds.max():.4f}, "
          f"전체 쌍 {full:.4f}, 차이(시드 평균 − 전체 쌍) {seeds.mean() - full:+.4f}")
out = PARTS.parent / 'pairwise_check_v1_summary.csv'
T.to_csv(out, index=False, encoding='utf-8-sig')
print('저장:', out)